# 00 · Orientation: what this code is, in MDO terms, and how to read it

This series takes you through `size_halo` **from the bottom up**: first the numerical machinery, then one physical
component at a time, then how components are joined, then the airframe, a single flight point, a mission, a coupled
sizing problem, and finally the 2,000-line Halo driver. By the end you should be able to open any file in
`src/aircraft_closure/` or `examples/halo_sizing.py` and say what every line contributes to the optimization
problem, and you should be able to add a component, a requirement or a mission segment yourself.

The series assumes you know multidisciplinary design optimization (MDO): what an NLP is, what MDF, IDF and
all-at-once formulations are, what KKT conditions and Lagrange multipliers mean. It does **not** assume you know
AeroSandbox or CasADi, or anything about this repository. Terms and symbols are collected in the [glossary](GLOSSARY.md).

**In this notebook**
1. The problem the tool solves, stated as an NLP.
2. Where it sits among MDO formulations, and the consequences of that choice.
3. The layers of the code and the rule that dependencies only point down.
4. A first complete sizing solve (2-3 seconds), so you know where the series is heading.
5. The reading plan for the rest of the series.

**Run time:** about 10 seconds.

In [1]:
from tutorial_helpers import *   # paths, asb / np / u / plt, plot style, check(), lb(), describe_opti()

## 1. The problem, as an NLP

The tool sizes an unmanned **series-hybrid-electric tiltrotor**: two turboshafts drive generators, the generators
and a battery feed a DC bus, electric motors on the bus turn two wing-tip proprotors through gearboxes. The question it
answers is the classic conceptual-design one:

> Given the payload, range, speed, hover, ceiling and failure requirements, what is the lightest aircraft
> (airframe + powertrain + energy) that meets all of them, and which requirement sizes each part?

Written as an optimization problem, the full Halo sizing is

$$
\begin{aligned}
\min_{x,\;y}\quad & m_\text{TO} \;/\; 1000 \\
\text{s.t.}\quad  & m_\text{TO} \,/\, \textstyle\sum_i m_i(x, y) = 1 && \text{(mass closure)}\\
                  & r_k(x, y_k) = 0 \quad \forall\ \text{flight points } k && \text{(trim, power balances, momentum theory, ...)}\\
                  & s_{j+1} = f_\text{SOC}(s_j, y_j),\ \ m_{j+1} = m_j - \dot m_{f,j}\,\Delta t_j && \text{(mission state chain)}\\
                  & g_\ell(x, y) \ge 0 \quad \forall\ \text{margins } \ell && \text{(every limit, normalized)}\\
                  & x_\text{lo} \le x \le x_\text{hi}
\end{aligned}
$$

- $x$ are **design variables**: take-off mass, wing area and position, tail areas, rotor disk area, solidity and
  tip speed, motor and generator ratings and speeds, gear ratio, battery strings, fuel load, heat-exchanger rating...
- $y$ are **operating (state) variables** of each flight point $k$: angle of attack, rotor speed, battery current,
  generator torque, battery share of the bus power, and a few coupling variables (rotor thrust with a blown wing,
  cooling drag).
- $r_k = 0$ are the **discipline equations** written as residuals: lift = weight, thrust = drag, the bus power balance,
  the battery's circuit equation, etc.
- $g_\ell \ge 0$ are **margins**, every limit normalized by its own value: motor torque, battery voltage, turboshaft
  power available, blade loading, static margin, state of charge...

There is **no inner solver and no outer loop.** Every discipline contributes expressions; IPOPT solves the KKT system
of the whole thing at once with exact first and second derivatives from CasADi's automatic differentiation.

## 2. Which MDO formulation this is, and why it matters to you

In the terminology of Martins and Lambe (AIAA J. 2013), this is an **all-at-once / simultaneous analysis and design
(SAND)** formulation:

| | MDF | IDF | **This code (AAO / SAND)** |
|---|---|---|---|
| Discipline analyses converged by | an inner MDA (e.g. Gauss-Seidel) per optimizer iteration | each discipline, given copies of coupling variables | **nobody: the optimizer** |
| Coupling variables (MTOM, bus voltage, induced velocity...) | outputs of the MDA | optimizer variables + consistency constraints | **optimizer variables** |
| Discipline state variables (alpha, rotor speed, currents) | internal to each analysis | internal to each analysis | **optimizer variables** |
| Discipline residuals | driven to zero inside the analyses | inside the analyses | **equality constraints of the NLP** |
| Feasible design at every iterate? | yes | multidisciplinary-feasible only at convergence | **only at convergence** |
| Derivatives | through the converged MDA (adjoint or FD) | per discipline | **exact, sparse, of explicit expressions** |

Consequences you will see in every file:

1. **Components never solve anything.** A component (motor, battery, rotor...) is a frozen dataclass whose methods
   return *expressions*. If a relation is implicit (the battery current that delivers a given power, the induced
   velocity of a rotor in power mode), the *caller* adds a variable and an equality. You will never find a `while`
   loop, a root finder or a fixed-point iteration in `src/`.
2. **Everything must be smooth and well scaled.** IPOPT is a Newton method on the KKT system. Variables get
   `scale=`; equalities are written as ratios of order one; margins are divided by their limits; `max()` becomes
   `softmax`. Tutorials 02-03 show why.
3. **Infeasibility is a result, not a crash of a loop.** When no aircraft closes, IPOPT reports an infeasible
   problem, where a weight loop would diverge or crawl.
4. **The Lagrange multipliers are the sensitivities.** Because the objective is take-off mass in tonnes and every
   margin is normalized, a multiplier reads directly as "tonnes of take-off mass per unit of margin". The binding
   constraints are printed by name.
5. **Local, not global.** IPOPT finds a local KKT point near its start. Starting points are managed explicitly
   (tutorial 20), never by iterating a quantity to convergence.

The topology layer of this code (tutorial 11) looks a bit like an OpenMDAO `Group` (`add`, `connect`), but it is much lighter: it describes the network
and never owns variables, execution order or solvers.

## 3. The layers of the code

```
examples/halo_sizing.py         THE aircraft: requirements, assumptions, design variables, the one Opti problem
examples/*.py                   the ladder of smaller problems this series walks through, plus validation cases
        │ uses
        ▼
src/aircraft_closure/
  mission/       segments -> flight conditions; the SOC / mass / fuel chain; cost per mission
  requirements/  capability requirements as flight conditions (hover, climb, speed, ceiling)
  performance/   build_flight_point: ONE operating point coupling aero + rotor + powertrain + thermal
  controls/      static margin, trim, directional stability, rudder authority
  aerodynamics/  simple polar, Scholz build-up, AeroBuildup wrapper, blown wing, trim drag, hover download
  vehicle/       wing, tails, fuselage, gear, systems, payload, fuel; Aircraft = mass and CG aggregation
  weights/       AFDD rotorcraft weight equations (rotor, drive, engine section, tiltrotor wing)
  thermal/       heat loads, lumped temperatures, heat-exchanger coupling
  powertrain/    components (motor, generator, battery, turboshaft, gearbox, rotor, cables...), port specs,
                 topologies (series hybrid builder), compatibility margins, redundancy, data loaders and fits
  core/          ports, topology (instances, buses, connections), normalized margins
        │ uses
        ▼
AeroSandbox (symbolic NumPy, Opti, atmosphere, AeroBuildup, Raymer weights) -> CasADi (AD, NLP) -> IPOPT
data/            CSV data: machine database, battery cell tables, JVX rotor data, engine deck, weight sources
```

**Dependencies point down only.** A component never imports a vehicle, a vehicle never imports a mission, nothing in
`src/` imports `examples/`. So every layer can be tested and understood on its own, which is exactly how this series
proceeds.

The rule that decides where code goes (`docs/CODING_CONVENTIONS.md`): *physical objects own their properties,
discipline models compute behaviour, assemblies compose components, and the caller owns optimization.* Let us check
the "nothing in `src/` imports `examples/`" claim directly:

In [2]:
import re
offenders = []
for path in (repo_root / "src").rglob("*.py"):
    text = path.read_text(encoding="utf-8")
    if re.search(r"^\s*(from|import)\s+examples", text, flags=re.M):
        offenders.append(path.relative_to(repo_root))
check("no module in src/ imports examples/", not offenders)

# And the size of each layer, to calibrate how much reading each tutorial covers.
sizes = {}
for path in sorted((repo_root / "src" / "aircraft_closure").rglob("*.py")):
    parts = path.relative_to(repo_root / "src" / "aircraft_closure").parts
    if len(parts) == 1:
        continue
    layer = parts[0]
    sizes[layer] = sizes.get(layer, 0) + len(path.read_text(encoding="utf-8").splitlines())
for layer, lines in sorted(sizes.items(), key=lambda item: -item[1]):
    print(f"{layer:<16}{lines:6d} lines")
print(f"{'examples/':<16}{sum(len(p.read_text(encoding='utf-8').splitlines()) for p in (repo_root / 'examples').glob('*.py')):6d} lines")

PASS  no module in src/ imports examples/
powertrain        2581 lines
export            1562 lines
trajectory         768 lines
aerodynamics       738 lines
vehicle            630 lines
performance        455 lines
mission            369 lines
core               317 lines
weights            261 lines
controls           144 lines
thermal            144 lines
requirements        85 lines
examples/         5506 lines


## 4. A first complete sizing solve

Before going to the bottom, look once at the top. `examples/halo_sizing.py` holds the whole aircraft. Its current
reference (every model at its highest fidelity) takes 10-15 minutes because it walks through a ladder of starting
points. It also keeps every earlier reference reproducible as a named assumption set. `assumptions_tier16` is
one of the simplest: simple aerodynamics, a constant-voltage battery, the Raymer wing weight, no thermal model, no
redundancy. It solves in a few seconds and has the same *structure* as the reference.

In [3]:
import time
from examples.halo_sizing import solve_halo_sizing, requirements_tier16, assumptions_tier16

time_start_s = time.perf_counter()
result = solve_halo_sizing(requirements_tier16, assumptions_tier16)
print(f"solved in {time.perf_counter() - time_start_s:.1f} s")
print(f"take-off mass   {result.mass_takeoff_kg:,.0f} kg ({lb(result.mass_takeoff_kg):,.0f} lb)")
print(f"empty mass      {result.mass_empty_kg:,.0f} kg;  fuel {result.mass_fuel_kg:,.0f} kg;  "
      f"payload {result.mass_payload_kg:,.0f} kg")
print(f"wing            {result.design.area_wing_m2:.1f} m2, span {result.span_m:.1f} m")
print(f"rotors          radius {result.radius_rotor_m:.2f} m, disk loading {result.disk_loading_kg_m2:.0f} kg/m2")
print(f"battery         {result.energy_battery_kWh:.0f} kWh;  cruise {result.velocity_cruise_m_s / u.knot:.0f} kt, "
      f"L/D {result.lift_to_drag_cruise:.1f}")
print(f"mass closure residual {result.closure_residual_kg:.1e} kg, smallest margin {result.min_margin:.1e}")
check("the fast set reproduces its documented 13,760 lb", abs(lb(result.mass_takeoff_kg) - 13760) < 5)
check("mass closes to round-off", abs(result.closure_residual_kg) < 1e-3)

solved in 1.9 s
take-off mass   6,242 kg (13,760 lb)
empty mass      4,308 kg;  fuel 1,034 kg;  payload 900 kg
wing            17.9 m2, span 10.5 m
rotors          radius 4.09 m, disk loading 59 kg/m2
battery         54 kWh;  cruise 156 kt, L/D 7.3
mass closure residual 9.1e-13 kg, smallest margin -1.0e-08
PASS  the fast set reproduces its documented 13,760 lb
PASS  mass closes to round-off


The most useful output is the list of **binding constraints**, the margins that are exactly zero at the optimum.
Each one names a flight condition and a limit. Together they say what sizes each part of the aircraft:

In [4]:
for label in result.binding:
    print("  ", label)

   max_speed: turboshaft power_shaft_W
   engine-out hover: generator power_shaft_W
   hover: propulsor power_shaft_W
   hover: motor power_shaft_W
   hover: gearbox power_input_W
   engine-out hover: battery discharge_power_W
   soc_after_engine_out_hover
   rotor_radius_m
   engine-out hover: turboshaft power_shaft_W
   engine-out hover: generator_gearbox power_input_W
   static_margin
   cn_beta_per_rad
   soc_end


Read one: `engine-out hover: battery discharge_power_W` means *in the engine-out hover flight point, the battery
is at its maximum discharge power*. So the battery is sized by that failure case, not by the mission energy. By
tutorial 18 you will have built a smaller problem of exactly this shape yourself, and by tutorial 20 you will know
where each of these lines comes from in the driver.

## Limits and next steps (at the tool level)

**What it can't do**

- It is **conceptual sizing**: quasi-steady points, handbook-calibrated weights, a calibrated analytic rotor. It sets targets and finds what drives the
  design; it does not certify anything. Wing strength at ultimate load, for one, is not yet demonstrated (tutorial 22).
- It is **local** optimization. IPOPT finds a local optimum from a starting point; the starting-point strategy (tutorial 20) makes that robust, not global.
- It has **no detailed geometry in the loop**: no internal layout, no CAD-level outer mould line. OpenVSP export and finite-element decks run afterwards as
  checks (`export/openvsp`), never inside the sizing.
- Performance is at the design mission only: no payload-range diagram yet.

**What's next**

The README frames the next steps as **work packages for specialist teams**, each replacing an input or a calibrated model with better data without
restructuring the model: supplier data for machines, cells and the engine; blade-element rotor maps; validated drag; Nastran strength, buckling and flutter;
linear models at the trim points and 6-DOF simulation; weight control with growth allowance carried inside the sizing (`docs/NEXT_STEPS.md`).

## 5. The reading plan

| # | Tutorial | What you can do afterwards |
|---|---|---|
| **Part 1: machinery** | | |
| 01 | AeroSandbox symbolic NumPy | write physics once for numbers and for variables; know what breaks a CasADi graph |
| 02 | `asb.Opti` in depth | variables, scaling, bounds, constraints, duals, solver output, failures, debugging |
| 03 | The sizing paradigm | weight loop vs all-at-once; smoothing; multipliers as sensitivities; local optima |
| 04 | Code conventions and margins | read any class in `src/`; the component contract; normalized margins |
| **Part 2: one component at a time** | | |
| 05 | Electric machines | McDonald losses, rubber machines, four mass models, sizing a motor to a duty |
| 06 | Gearboxes and the turboshaft | stage counts, lapse, part-power fuel curves from an engine deck |
| 07 | Batteries | constant-voltage pack, equivalent circuit (OCV, R0 + 2 RC), the two-root power equation |
| 08 | Rotors | actuator disk in thrust and power mode; momentum + profile rotor calibrated on JVX |
| 09 | Electrical and thermal parts | cables, contactors, inverters, DC/DC, heat exchanger, lumped temperatures |
| **Part 3: connecting components** | | |
| 10 | A coupled point by hand | six components, one Opti, every connection written explicitly |
| 11 | Ports, topology, margins | the same point from a network description; redundancy and degraded states |
| **Part 4: the airframe** | | |
| 12 | Airframe components, mass and CG | surfaces, fuselage, items, installation, `Aircraft`; mass closure as an equality |
| 13 | Aerodynamics | simple, Scholz and AeroBuildup models; blown wing, trim drag, download; cruise closure |
| 14 | Stability and tail sizing | neutral point, trim, Cn_beta, failed-rotor rudder, tails inside the closure |
| **Part 5: flying it** | | |
| 15 | The flight point | `build_flight_point` line by line: hover and airplane mode |
| 16 | Capability requirements | requirement points at MTOM, sizing the powertrain to them |
| 17 | Missions | segments, the SOC and mass chain, prescribed vs free energy split |
| 18 | Coupled sizing | aircraft + mission + requirements + reserves + stability in one problem |
| **Part 6: the Halo** | | |
| 19 | The Halo driver | requirements, assumptions, design variables, building and solving the Halo |
| 20 | Studies, starts and extensions | starting points, sensitivities, objectives, adding your own model |
| **After sizing** | | |
| 21 | Trim, corridor and trajectories | the conversion corridor and direct-collocation trajectories on the sized aircraft |
| 22 | Weights in depth | the stiffness-sized tiltrotor wing and its validation; the growth factor as a multiplier; growth allowance and uncertainty of the empty weight |

Every notebook ends with `check_summary()`: the `check(...)` lines verify the claims made in the text, so a
notebook that still passes still tells the truth about the code.

In [5]:
check_summary()

3 of 3 checks passed
